In [ ]:
!unzip imagesX.zip

Archive:  imagesX.zip
  inflating: imagesX/Photo on 09-12-2025 at 10.40 AM #2.jpg  
replace __MACOSX/imagesX/._Photo on 09-12-2025 at 10.40 AM #2.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import os
import unicodedata # Import unicodedata for filename normalization

# Define the variables needed for the classification script
base_dir = "imagesX"
subfolders = ["Non-Cheating", "Cheating", "DifferentX"]

# Create target directories if they don't exist
for folder in subfolders:
    path = os.path.join(base_dir, folder)
    os.makedirs(path, exist_ok=True) # Ensure these folders exist for shutil.move later
    print(f"Ensured directory: {path}")

# List all the images you uploaded for processing (assuming they are directly in imagesX)
# Normalize filenames to NFC form immediately to handle potential unicode inconsistencies
raw_images = [unicodedata.normalize('NFC', f) for f in os.listdir(base_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
all_images = []

print("\n--- Normalizing image filenames ---")
for image_file_nfc in raw_images:
    original_full_path = os.path.join(base_dir, image_file_nfc)
    target_file_name = unicodedata.normalize('NFC', image_file_nfc.replace('\u202f', ' ')) # Ensure regular spaces and NFC
    target_full_path = os.path.join(base_dir, target_file_name)

    if image_file_nfc != target_file_name:
        # Rename if the filename needs correction (e.g., has \u202f or different normalization)
        if os.path.exists(original_full_path):
            try:
                os.rename(original_full_path, target_full_path)
                print(f"Renamed '{image_file_nfc}' to '{target_file_name}' (NFC normalized with regular spaces).")
                all_images.append(target_file_name)
            except OSError as e:
                print(f"ERROR: Failed to rename '{original_full_path}' to '{target_full_path}': {e}. Appending original NFC name.")
                all_images.append(image_file_nfc) # Fallback to original NFC name if rename fails
        else:
            print(f"WARNING: Original file '{original_full_path}' not found for renaming. Appending original NFC name.")
            all_images.append(image_file_nfc)
    else:
        # No rename needed, or already in desired NFC with regular spaces
        all_images.append(image_file_nfc)

print(f"\nFound {len(all_images)} images to process after normalization. Proceeding with YOLO setup...")

In [ ]:
# Install the ultralytics library for YOLOv8
!pip install ultralytics

from ultralytics import YOLO
import shutil

# Load a pre-trained YOLOv8 model (small version)
# This model is trained on the COCO dataset, which includes 'person' (for face/head) and many common objects.
model = YOLO('yolov8s.pt')

In [ ]:
import os
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np # Needed for plotting

# Load YOLO again for validation
model = YOLO("yolov8n.pt")

# --- FIX: Use the correct, case-sensitive folder name ---
base = "/content/ImagesX"
# --------------------------------------------------------

folders = ["cheating", "non-cheating", "differentX"]

# Counters for tracking results
correct = 0
incorrect = 0
mistakes = []

FACE_KEYWORDS = ["face", "person"]
OBJECT_KEYWORDS = ["cell phone", "book", "cup", "bottle", "hand", "remote"]

def contains_face(results):
    for r in results:
        for box in r.boxes:
            label = r.names[int(box.cls)]
            if label in FACE_KEYWORDS:
                return True
    return False

def contains_object(results):
    for r in results:
        for box in r.boxes:
            label = r.names[int(box.cls)]
            if label in OBJECT_KEYWORDS:
                return True
    return False

# Loop over all folders (where the image's folder is the Ground Truth label)
print(f"Starting verification using {base}...")
for folder in folders:
    # This creates the CORRECT path, e.g., /content/ImagesX/cheating
    folder_path = os.path.join(base, folder)

    # CHECK: Make sure the path exists before attempting to list files
    if not os.path.exists(folder_path):
        print(f"Error: Directory not found. Check capitalization: {folder_path}")
        continue

    # This line should now work:
    for img in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img)

        if os.path.isdir(img_path) or not img.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        # YOLO Prediction
        results = model(img_path, verbose=False) # Use verbose=False to keep output clean
        face = contains_face(results)
        obj = contains_object(results)

        # YOLO predicted class based on Part 2 logic
        if face and obj:
            pred = "cheating"
        elif face and not obj:
            pred = "non-cheating"
        elif obj and not face:
            pred = "differentX"
        else:
            # If neither face nor specified object is detected, prediction is "unknown"
            pred = "unknown"

        # Compare with ground truth (which is the 'folder' name)
        if pred == folder:
            correct += 1
        else:
            incorrect += 1
            # Store the misplaced image details
            mistakes.append((img, folder, pred))

## Results and Markdown Output
print("\n" + "="*50)
print("YOLO Verification Results:")
print("="*50)
print(f"Total images processed: {correct + incorrect}")
print(f"Correct classifications: {correct}")
print(f"Incorrect classifications: {incorrect}")

# --- Markdown Table Output (Part 3 Requirement) ---
print("\n### Annotation Verification Details (for Jupyter Markdown)")
print("| Image File | Correct Label | YOLO Predicted Label | Note |")
print("| :--- | :---: | :---: | :--- |")
for m in mistakes:
    # Example Note: 'Misplaced: Should be cheating'
    note = f"Misplaced: Should be {m[1]}"
    print(f"| {m[0]} | **{m[1]}** | {m[2]} | {note} |")
print("\n---")
print(f"\nOverall Accuracy: {correct / (correct + incorrect) * 100:.2f}%")
print("---")

## Bar Chart Generation (Part 3 Requirement)
labels = ['Correct', 'Incorrect']
counts = [correct, incorrect]
colors = ['green', 'red']

plt.figure(figsize=(7, 5))
plt.bar(labels, counts, color=colors)
plt.title('YOLO Annotation Verification Summary')
plt.ylabel('Number of Images')
# Add count labels on top of the bars
for i, count in enumerate(counts):
    plt.text(i, count + 0.5, str(count), ha='center')

plt.show()
#



--- Starting Image Classification (With Object Tracking) ---
DEBUG: Found AirPods (Remote/Misc) with person in Photo on 09-12-2025 at 10.39 AM.jpg.
Moved Photo on 09-12-2025 at 10.39 AM.jpg to **Cheating**
DEBUG: Found AirPods (Remote/Misc), iPhone (Cell Phone) with person in WhatsApp Image 2025-12-09 at 10.31.39 AM.jpeg.
Moved WhatsApp Image 2025-12-09 at 10.31.39 AM.jpeg to **Cheating**
DEBUG: Found AirPods (Remote/Misc) with person in Photo on 09-12-2025 at 10.39 AM #2.jpg.
Moved Photo on 09-12-2025 at 10.39 AM #2.jpg to **Cheating**
DEBUG: Found AirPods (Remote/Misc) with person in Photo on 09-12-2025 at 10.39 AM #3.jpg.
Moved Photo on 09-12-2025 at 10.39 AM #3.jpg to **Cheating**
DEBUG: Found AirPods (Remote/Misc) with person in Photo on 09-12-2025 at 10.40 AM #2.jpg.
Moved Photo on 09-12-2025 at 10.40 AM #2.jpg to **Cheating**
DEBUG: Found AirPods (Remote/Misc) with person in WhatsApp Image 2025-12-09 at 10.31.38 AM.jpeg.
Moved WhatsApp Image 2025-12-09 at 10.31.38 AM.jpeg to **

FileNotFoundError: imagesX/Photo on 09-12-2025 at 10.40 AM #2.jpg does not exist

In [ ]:
print("\n--- Final Folder Contents ---")
for folder in subfolders:
    folder_path = os.path.join(base_dir, folder)
    if os.path.isdir(folder_path):
        count = len(os.listdir(folder_path))
    else:
        count = 0 # If the folder doesn't exist, it contains 0 images
    print(f"Folder '{folder}': {count} images")

In [ ]:
# Install necessary libraries
!pip install Pillow imgaug

from PIL import Image
import numpy as np
import imgaug.augmenters as iaa
import os
import shutil

# --- Setup Augmentation Folders ---
AUG_BASE_DIR = "AugImagesX"
aug_subfolders = ["non-cheating", "cheating", "differentX"]

# Create the new base directory
os.makedirs(AUG_BASE_DIR, exist_ok=True)

# Create the subdirectories and copy original images
print(f"Creating augmentation structure in: {AUG_BASE_DIR}")

# Copy original images to the new augmented folder structure
for folder in subfolders:
    src_dir = os.path.join(base_dir, folder)
    dst_dir = os.path.join(AUG_BASE_DIR, folder)
    os.makedirs(dst_dir, exist_ok=True)

    # Copy original images
    for image_name in os.listdir(src_dir):
        if image_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            shutil.copy(os.path.join(src_dir, image_name), dst_dir)

print("Original images copied successfully to AugImagesX subfolders.")

# Define 5 Augmentation Methods using imgaug Sequential
# 1. Flip (Horizontal)
# 2. Rotation (Small angle)
# 3. Gaussian Blur
# 4. Sharpen
# 5. Contrast Change (Multiply)

# Create a sequence of augmenters
augmenters = {
    "flip_h": iaa.Fliplr(1.0),                                  # 1. Horizontal Flip
    "rotate_10": iaa.Affine(rotate=(-10, 10), name="rotate"),   # 2. Rotation up to 10 degrees
    "blur_g": iaa.GaussianBlur(sigma=(0.0, 1.0)),               # 3. Light Gaussian Blur
    "sharpen": iaa.Sharpen(alpha=(0.0, 1.0), lightness=(0.75, 1.5)), # 4. Sharpen
    "contrast": iaa.Multiply((0.5, 1.5))                        # 5. Contrast/Brightness adjustment
}

# --- Apply Augmentation to all Images ---

print("\n--- Starting Data Augmentation ---")

# Iterate through original folders (non-cheating, cheating, differentX)
for folder in aug_subfolders:
    folder_path = os.path.join(AUG_BASE_DIR, folder)
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    for image_name in image_files:
        if image_name.startswith("aug_"):
            continue # Skip already augmented images

        input_path = os.path.join(folder_path, image_name)

        try:
            # Load image using PIL, convert to NumPy array for imgaug
            img = np.array(Image.open(input_path).convert("RGB"))
        except Exception as e:
            print(f"Error loading {image_name}: {e}")
            continue

        # Apply each of the 5 augmentation methods
        for aug_name, augmenter in augmenters.items():
            # Augment the image
            aug_img = augmenter.augment_image(img)

            # Convert back to PIL Image and save
            aug_img_pil = Image.fromarray(aug_img)
            new_file_name = f"aug_{aug_name}_{image_name}"
            output_path = os.path.join(folder_path, new_file_name)

            aug_img_pil.save(output_path)

    print(f"Applied 5 augmentations to images in '{folder}'. New files generated.")

print("--- Data Augmentation Complete ---")

In [ ]:
# Install necessary libraries for ViT
!pip install transformers datasets accelerate pytorch-lightning

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from transformers import ViTForImageClassification, ViTImageProcessor
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# Ensure all folders (including augmented ones) are available for the dataset
# We will use the 'AugImagesX' folder as the final dataset path
DATA_DIR = AUG_BASE_DIR
NUM_CLASSES = len(aug_subfolders) # Should be 3: non-cheating, cheating, differentX

# --- 3A. Data Transformations and DataLoader ---

# Load the ViT processor to get the required normalization and image size
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
image_size = processor.size["height"]

# Define transformations: ViT requires resizing and normalization
# We only use the base transformations here, as augmentation was done offline
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

# Create Dataset and DataLoader
# Use ImageFolder structure (which matches your AugImagesX/subfolder setup)
full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

# Split data into Training and Validation sets (e.g., 80/20 split)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

print(f"Dataset Size: Total={len(full_dataset)}, Train={train_size}, Validation={val_size}")
print(f"Classes: {full_dataset.classes}")
#

# --- 3B. ViT Model and Training Module ---

class ViTClassifier(pl.LightningModule):
    def __init__(self, num_labels, learning_rate=2e-5):
        super().__init__()
        # Load pre-trained ViT model and set the number of classes
        self.model = ViTForImageClassification.from_pretrained(
            'google/vit-base-patch16-224',
            num_labels=num_labels,
            ignore_mismatched_sizes=True # Allow label mismatch for fine-tuning
        )
        self.learning_rate = learning_rate
        self.criterion = torch.nn.CrossEntropyLoss()
        self.save_hyperparameters()

    def forward(self, pixel_values):
        return self.model(pixel_values=pixel_values).logits

    def training_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = self.criterion(logits, labels)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = self.criterion(logits, labels)
        self.log('val_loss', loss, prog_bar=True)

        # Calculate accuracy
        preds = torch.argmax(logits, dim=1)
        correct = (preds == labels).sum().item()
        accuracy = correct / len(labels)
        self.log('val_acc', accuracy, prog_bar=True)

        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate)
        return optimizer

# --- 3C. Training Execution ---

# Initialize the model
vit_model = ViTClassifier(num_labels=NUM_CLASSES)

# Define callbacks for saving the best model and stopping if no improvement
checkpoint_callback = ModelCheckpoint(
    monitor='val_acc',
    mode='max',
    dirpath='vit_checkpoints',
    filename='best-vit-model',
    save_top_k=1
)

early_stop_callback = EarlyStopping(
    monitor='val_loss',
    min_delta=0.00,
    patience=3, # Stop after 3 epochs with no improvement in validation loss
    verbose=False,
    mode='min'
)

# Initialize the Trainer
trainer = pl.Trainer(
    max_epochs=10, # Start with 10 epochs
    accelerator='auto', # Use GPU if available
    callbacks=[checkpoint_callback, early_stop_callback]
)

print("\n--- Starting ViT Model Fine-Tuning ---")
trainer.fit(vit_model, train_loader, val_loader)
print("--- ViT Model Training Complete ---")

# You now have the best model saved in 'vit_checkpoints/best-vit-model.ckpt'